In [4]:
"""
Beige Book Scraper — 2020 to present (June 2026)
================================================
Scrapes all regional Beige Book reports from the Federal Reserve Bank
of Minneapolis public archive. Saves each report as a .txt file and
produces a combined CSV of all text.

Output structure:
    out/txt/{year}/{month:02d}/{year}-{month:02d}-{region}.txt
    out/csv/beige_book_2020_2026.csv
    out/csv/missing.csv

Usage:
    pip install requests beautifulsoup4 html5lib
    python beige_book_scraper.py
"""

import os
import re
import csv
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

START_YEAR = 2020
END_YEAR   = 2026          # inclusive
END_MONTH  = 6             # June 2026 is the latest published as of today

BASE_URL = "https://www.minneapolisfed.org/beige-book-reports"

# Region codes — same as the original tools.py gen() used
REGIONS = ["su", "bo", "ny", "ph", "cl", "ri", "at", "ch", "sl", "mi", "kc", "da", "sf"]

# Beige Book is published 8× per year.  These are the typical release months.
# (Jan, Mar, Apr, Jun, Jul, Sep, Oct, Nov — but months shift slightly each year.)
# Rather than hard-code exact dates, we attempt ALL months and skip 404s.
ALL_MONTHS = list(range(1, 13))

DELAY_SECONDS = 0.5   # be polite to the server


# ---------------------------------------------------------------------------
# File helpers  (replaces tools.py)
# ---------------------------------------------------------------------------

def get_txt_path(year: int, month: int, region: str) -> str:
    directory = os.path.join("out", "txt", str(year), f"{month:02d}")
    os.makedirs(directory, exist_ok=True)
    return os.path.join(directory, f"{year}-{month:02d}-{region}.txt")


def build_url(year: int, month: int, region: str) -> str:
    return f"{BASE_URL}/{year}/{year}-{month:02d}-{region}"


def iter_targets():
    """Yield (year, month, region) for every combination to attempt."""
    now = datetime.now()
    for year in range(START_YEAR, END_YEAR + 1):
        for month in ALL_MONTHS:
            # Don't go past today's month
            if year == END_YEAR and month > END_MONTH:
                break
            for region in REGIONS:
                yield year, month, region


# ---------------------------------------------------------------------------
# Scraping helpers
# ---------------------------------------------------------------------------

def fetch_page(url: str):
    """Return a BeautifulSoup object or raise ValueError on 404."""
    resp = requests.get(url, timeout=15)
    if resp.status_code == 404:
        raise ValueError(f"404: {url}")
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html5lib")


def extract_text(soup: BeautifulSoup) -> str:
    """Pull the main article text from the Minneapolis Fed page."""
    # Primary container used by the Minneapolis Fed template
    div = soup.find("div", class_="col-sm-12 col-lg-8 offset-lg-1")
    if div is None:
        # Fallback: try the article tag
        div = soup.find("article") or soup.find("main")
    if div is None:
        raise ValueError("Could not locate article content in page.")

    raw = re.sub(r"\s*\n\s*", "\n", div.get_text()).strip()
    # The first few lines are typically the title/date header — skip them
    lines = raw.split("\n")
    # Drop lines until we hit substantive content (heuristic: skip first 3)
    content = "\n".join(lines[3:]).strip()
    return content


def extract_date_line(soup: BeautifulSoup) -> str:
    """Extract the publication date line (e.g. 'January 14, 2026')."""
    div = soup.find("div", class_="col-sm-12 col-lg-8 offset-lg-1")
    if div is None:
        return ""
    raw = re.sub(r"\s*\n\s*", "\n", div.get_text()).strip()
    lines = raw.split("\n")
    return lines[2] if len(lines) > 2 else ""


# ---------------------------------------------------------------------------
# Main scraper
# ---------------------------------------------------------------------------

def scrape():
    os.makedirs(os.path.join("out", "csv"), exist_ok=True)

    csv_rows = []        # collected for combined CSV output
    missing  = []        # (year, month, region) tuples that returned 404

    targets = list(iter_targets())
    total   = len(targets)

    print(f"Starting scrape: {total} combinations to check")
    print(f"Range: {START_YEAR}-01 → {END_YEAR}-{END_MONTH:02d}, all 13 regions\n")

    for i, (year, month, region) in enumerate(targets, 1):
        url  = build_url(year, month, region)
        label = f"[{i:>5}/{total}] {year}-{month:02d} {region:>2}"

        try:
            soup = fetch_page(url)
            text = extract_text(soup)
            date_line = extract_date_line(soup)

            # Save individual .txt file
            txt_path = get_txt_path(year, month, region)
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(text)

            # Accumulate for CSV
            csv_rows.append({
                "year":      year,
                "month":     f"{month:02d}",
                "region":    region,
                "date_line": date_line,
                "url":       url,
                "text":      text.replace("\n", " "),
            })

            print(f"{label}  ✓  ({len(text):,} chars)")

        except ValueError:
            # 404 — this month simply wasn't a Beige Book release month
            missing.append((year, month, region))
            print(f"{label}  — (not published)")

        except Exception as exc:
            print(f"{label}  ERROR: {exc}")
            missing.append((year, month, region))

        time.sleep(DELAY_SECONDS)

    # ------------------------------------------------------------------
    # Write combined CSV
    # ------------------------------------------------------------------
    csv_path = os.path.join("out", "csv", "beige_book_2020_2026.csv")
    fieldnames = ["year", "month", "region", "date_line", "url", "text"]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_rows)

    print(f"\n✅  Saved {len(csv_rows)} reports → {csv_path}")

    # ------------------------------------------------------------------
    # Write missing log
    # ------------------------------------------------------------------
    missing_path = os.path.join("out", "csv", "missing.csv")
    with open(missing_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["year", "month", "region"])
        for year, month, region in missing:
            writer.writerow([year, f"{month:02d}", region])

    print(f"📋  Missing/skipped log → {missing_path}")
    print(f"\nDone. {len(csv_rows)} successful, {len(missing)} skipped.")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    scrape()

Starting scrape: 1014 combinations to check
Range: 2020-01 → 2026-06, all 13 regions

[    1/1014] 2020-01 su  — (not published)
[    2/1014] 2020-01 bo  — (not published)
[    3/1014] 2020-01 ny  — (not published)
[    4/1014] 2020-01 ph  — (not published)
[    5/1014] 2020-01 cl  — (not published)


KeyboardInterrupt: 